# Setup: Unity Catalog Volume + Genie Space

This notebook creates:

1. **Unity Catalog volume** under `melissap.melissa_pang` for storing project files.
2. **Genie space** for natural-language SQL exploration over your Unity Catalog tables.

**Prerequisites:** `databricks-sdk` installed; Databricks workspace authentication (e.g. `~/.databrickscfg` profile or `DATABRICKS_HOST` / `DATABRICKS_TOKEN` env vars).

In [13]:
# Configuration
CATALOG = "melissap"
SCHEMA = "melissa_pang"
VOLUME_NAME = "project_volume"
GENIE_SPACE_TITLE = "build-con-mp"
GENIE_SPACE_DESCRIPTION = "Natural language SQL exploration"

# Optional: set a SQL warehouse ID, or leave None to auto-pick a running warehouse
WAREHOUSE_ID = "e9b34f7a2e4b0561"

# Optional: table identifiers for Genie (e.g. ["melissap.melissa_pang.my_table"])
# If the serialized_space template does not support tables, add them in the Genie UI after creation.
TABLE_IDENTIFIERS = []  # e.g. ["melissap.melissa_pang.sample_trips_e2_demo_field_eng"]

# Optional: clone from an existing Genie space to reuse its layout (set to space_id string or None)
TEMPLATE_SPACE_ID = None

In [6]:
from databricks.sdk import WorkspaceClient

# Use profile for local runs (aligns with create_schema.py); on Databricks use WorkspaceClient() for runtime auth
w = WorkspaceClient(profile="DEFAULT")

In [7]:
# Create Unity Catalog volume if it does not exist
from databricks.sdk.service.catalog import VolumeType

existing = [v for v in w.volumes.list(catalog_name=CATALOG, schema_name=SCHEMA) if v.name == VOLUME_NAME]
if existing:
    print(f"Volume already exists: {CATALOG}.{SCHEMA}.{VOLUME_NAME}")
else:
    vol = w.volumes.create(
        catalog_name=CATALOG,
        schema_name=SCHEMA,
        name=VOLUME_NAME,
        volume_type=VolumeType.MANAGED,
        comment="Project volume for file storage",
    )
    print(f"Created volume: {vol.full_name}")

Volume already exists: melissap.melissa_pang.project_volume


In [9]:
# Resolve SQL warehouse: use WAREHOUSE_ID if set, otherwise pick first running warehouse
if WAREHOUSE_ID:
    warehouse_id = WAREHOUSE_ID
    print(f"Using configured warehouse: {warehouse_id}")
else:
    warehouses = list(w.warehouses.list())
    running = [wh for wh in warehouses if getattr(wh, "state", None) == "RUNNING"]
    if not running:
        raise RuntimeError(
            "No running SQL warehouse found. Start a warehouse or set WAREHOUSE_ID in the config cell."
        )
    warehouse_id = running[0].id
    print(f"Using warehouse: {running[0].name} ({warehouse_id})")

Using configured warehouse: e9b34f7a2e4b0561


In [14]:
# Create Genie space
import json

if TEMPLATE_SPACE_ID:
    # Clone from existing space: get its serialized_space and create a new space with our title/description
    template = w.genie.get_space(space_id=TEMPLATE_SPACE_ID, include_serialized_space=True)
    serialized_space = template.serialized_space or "{}"
    print(f"Using serialized_space from template space {TEMPLATE_SPACE_ID}")
else:
    # Minimal serialized_space placeholder. For a full layout (tables, instructions), create a space in the
    # Genie UI once, then use Get Genie Space API to export serialized_space, or set TEMPLATE_SPACE_ID above.
    serialized_space = json.dumps({"version": 1})

space = w.genie.create_space(
    warehouse_id=warehouse_id,
    serialized_space=serialized_space,
    title=GENIE_SPACE_TITLE,
    description=GENIE_SPACE_DESCRIPTION or None,
)
print(f"Created Genie space: {getattr(space, 'title', GENIE_SPACE_TITLE)} ")
print(f"Open in workspace: AI/BI Genie > find space by title '{GENIE_SPACE_TITLE}'")

Created Genie space: build-con-mp 
Open in workspace: AI/BI Genie > find space by title 'build-con-mp'
